In [ ]:
!pip -q install rdkit==2026.03.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 21.3 MB/s eta 0:00:00


In [ ]:
import re
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import inchi

In [ ]:
# Load the CSV
df = pd.read_csv("nat_uv_db.csv")
# Preview
print(df.columns)

Index(['Canonical_SMILES', 'NatUV ID', 'Common name', 'PubChem ID',
       'ChEMBL ID', 'Isomeric SMILES', 'Kingdom', 'Genus', 'Species',
       'Mexican State', 'Location', 'Reference', 'Journal', 'DOI', 'Year',
       'Spasmolytic', 'Antinociceptive', 'Anti-inflammatory', 'Antimicrobial',
       'Anticancer / Citotoxic', 'Antihypertensive', 'Insecticide',
       'Herbicide', 'Antioxidant', 'Cardiovascular', 'Antiallergic',
       'Metabolism', 'Antiparasitic', 'Neurological diseases/targets',
       'Antiplatelet/Anticoagulant', 'Antiviral', 'Immunomodulator'],
      dtype='object')


In [ ]:
len(df)

227

In [ ]:
# Filter to fungal origin only
df = df[df["Kingdom"].astype(str).str.strip().str.lower() == "fungi"].copy()
print(df.shape)

(27, 32)


In [ ]:
bioactivity_columns=['Spasmolytic', 'Antinociceptive', 'Anti-inflammatory', 'Antimicrobial',
       'Anticancer / Citotoxic', 'Antihypertensive', 'Insecticide',
       'Herbicide', 'Antioxidant', 'Cardiovascular', 'Antiallergic',
       'Metabolism', 'Antiparasitic', 'Neurological diseases/targets',
       'Antiplatelet/Anticoagulant', 'Antiviral', 'Immunomodulator']

In [ ]:
# Count bioactivities

bio_cols = bioactivity_columns
print("\nCounts per bioactivity:\n")
for col in bio_cols:
    print(f"{col} -> {int(df[col].sum())} compounds")


Counts per bioactivity:

Spasmolytic -> 0 compounds
Antinociceptive -> 1 compounds
Anti-inflammatory -> 5 compounds
Antimicrobial -> 10 compounds
Anticancer / Citotoxic -> 13 compounds
Antihypertensive -> 0 compounds
Insecticide -> 1 compounds
Herbicide -> 0 compounds
Antioxidant -> 8 compounds
Cardiovascular -> 3 compounds
Antiallergic -> 1 compounds
Metabolism -> 9 compounds
Antiparasitic -> 7 compounds
Neurological diseases/targets -> 7 compounds
Antiplatelet/Anticoagulant -> 1 compounds
Antiviral -> 11 compounds
Immunomodulator -> 2 compounds


In [ ]:
# Remove rows where all bioactivities are 0
existing_columns = [c for c in bioactivity_columns if c in df.columns]
# Remove rows where all bioactivity columns are 0
df = df[df[existing_columns].any(axis=1)]
len(df)

15

In [ ]:
df = df.drop(columns=[c for c in existing_columns if df[c].sum() == 0])

# Get the remaining bioactivity columns
remaining = [c for c in df.columns if c in existing_columns]
print("Remaining bioactivity columns:", remaining)
len(remaining)

Remaining bioactivity columns: ['Antinociceptive', 'Anti-inflammatory', 'Antimicrobial', 'Anticancer / Citotoxic', 'Insecticide', 'Antioxidant', 'Cardiovascular', 'Antiallergic', 'Metabolism', 'Antiparasitic', 'Neurological diseases/targets', 'Antiplatelet/Anticoagulant', 'Antiviral', 'Immunomodulator']


14

In [ ]:
df.shape

(15, 29)

In [ ]:
df.columns

Index(['Canonical_SMILES', 'NatUV ID', 'Common name', 'PubChem ID',
       'ChEMBL ID', 'Isomeric SMILES', 'Kingdom', 'Genus', 'Species',
       'Mexican State', 'Location', 'Reference', 'Journal', 'DOI', 'Year',
       'Antinociceptive', 'Anti-inflammatory', 'Antimicrobial',
       'Anticancer / Citotoxic', 'Insecticide', 'Antioxidant',
       'Cardiovascular', 'Antiallergic', 'Metabolism', 'Antiparasitic',
       'Neurological diseases/targets', 'Antiplatelet/Anticoagulant',
       'Antiviral', 'Immunomodulator'],
      dtype='object')

In [ ]:
# Remove rows where SMILES is missing in meta file
before = len(df)
df = df[df["Canonical_SMILES"].notna()].reset_index(drop=True)
after = len(df)

print(f"Removed {before - after} compounds with no SMILES.")
print(f"Final dataset size: {after} compounds")

Removed 0 compounds with no SMILES.
Final dataset size: 15 compounds


In [ ]:
# Function: SMILES -> canonical SMILES
def canonicalize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    except:
        return None  # invalid SMILES

In [ ]:
# Function: SMILES -> InChIKey
def smiles_to_inchikey(smi):
    try:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            return None
        return inchi.MolToInchiKey(mol)
    except:
        return None

In [ ]:
# Compute canonical smiles
df["canonical_smiles"] = df["Canonical_SMILES"].apply(canonicalize_smiles)
# Report how many succeeded
success_CS = df["canonical_smiles"].notna().sum()
total = len(df)
print(f"Generated canonical smiles for {success_CS}/{total} compounds ({success_CS/total*100:.1f}%).")

Generated canonical smiles for 15/15 compounds (100.0%).


In [ ]:
# Compute InChIKeys
df["InChIKey"] = df["Canonical_SMILES"].apply(smiles_to_inchikey)
# Report how many succeeded
success_InChI = df["InChIKey"].notna().sum()
total = len(df)
print(f"Generated InChIKeys for {success_InChI}/{total} compounds ({success_InChI/total*100:.1f}%).")

Generated InChIKeys for 15/15 compounds (100.0%).


In [ ]:
df_out = df[['Canonical_SMILES', 'NatUV ID', 'Common name', 'PubChem ID',
       'Species','DOI',
       'Antinociceptive', 'Anti-inflammatory', 'Antimicrobial',
       'Anticancer / Citotoxic', 'Insecticide', 'Antioxidant',
       'Cardiovascular', 'Antiallergic', 'Metabolism', 'Antiparasitic',
       'Neurological diseases/targets', 'Antiplatelet/Anticoagulant',
       'Antiviral', 'Immunomodulator', 'canonical_smiles', 'InChIKey']]

In [ ]:
df_out.to_csv("nat_uv_db_processed.csv", index=False)